In [54]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [55]:
# Constants
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2

In [56]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [57]:
target_col = 'label'
# First set of features (Model explainability with log regression)
feature_cols_1 = ['vdop', 'ax', 'hdop', 'cog_rad', 'hdop', 'jamming_indicator', 'terrain_alt_valid', 'dist_bottom_valid', 'terrain_alt', 'lat_x', 'lat_x', 'lat_y']
X1 = df[feature_cols_1]
y = df[target_col]

# Label encode if needed
if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

# Second set of features (From Xgboost model with all features)
feature_cols_2 = ['lat_y', 'vel_e_m_s', 'vel_n_m_s', 'c_variance_rad', 'hdop', 's_variance_m_s', 'q[0]', 'vel_m_s' ,'cog_rad', 'q[3]']
X2 = df[feature_cols_2]

# Split both sets with the same random state
X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)
X2_train, X2_test, _, _ = train_test_split(
    X2, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)  # y_train/y_test already made above

# Scale both sets
scaler1 = StandardScaler()
X1_train_scaled = scaler1.fit_transform(X1_train)
X1_test_scaled = scaler1.transform(X1_test)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

In [58]:
# Train and evaluate first model
model1 = XGBClassifier(
    objective='binary:logistic',
    random_state=RANDOM_STATE,
    eval_metric='mae',
)

model1.fit(X1_train_scaled, y_train)
y1_pred = model1.predict(X1_test_scaled)
accuracy1 = accuracy_score(y_test, y1_pred)
print(f"Model 1 (Model explainability) Accuracy: {accuracy1 * 100:.2f}%")
print("Classification report (Model 1):")
print(classification_report(y_test, y1_pred, zero_division=0))

Model 1 (Model explainability) Accuracy: 99.79%
Classification report (Model 1):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [59]:
# Train and evaluate second model
model2 = XGBClassifier(
    objective='binary:logistic',
    random_state=RANDOM_STATE,
    eval_metric='mae',
)
model2.fit(X2_train_scaled, y_train)
y2_pred = model2.predict(X2_test_scaled)
accuracy2 = accuracy_score(y_test, y2_pred)
print(f"Model 2 (Xgboost features) Accuracy: {accuracy2 * 100:.2f}%")
print("Classification report (Model 2):")
print(classification_report(y_test, y2_pred, zero_division=0))

Model 2 (Xgboost features) Accuracy: 99.79%
Classification report (Model 2):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898

